# Globus over the Cape Town path: a demonstration

The weave built one Globus Connect Server endpoint per site (FIU, CIEN). Each VM has **two** addresses that matter:

| address | used for | who connects to it |
|---|---|---|
| public (FABnetv4Ext or AmLight L3 port) | registration, Globus control (TCP 443) | the Globus cloud service |
| data NIC (L2 facility-port VLAN) | **GridFTP data** (TCP 50000-51000), set as GCS `--data-interface` | the Cape Town DTN |

```
                     Globus cloud (transfer.api.globus.org)
               443 /                                   \  outbound only
                  v                                     \
   FABRIC DTN (FIU / CIEN)                         Cape Town DTN (SANReN)
     public IP  <- control                          Globus Connect Personal
     data IP    <======== data, opened by Cape Town ========  VLAN 1400 / 2901
                 (same L2 subnet via the facility port)
```

**Why the data takes the VLAN:** Globus tells the Cape Town endpoint *which address to send data to* - the DTN's
advertised data interface. Globus Connect Personal always opens data connections itself, and its OS routing sends
anything for that address out the VLAN. Nothing depends on where the transfer is submitted: the CLI and the web app
only hand the request to the Globus cloud, and the bytes never pass through the submitting machine.

This notebook:
1. checks the data path from each DTN (route, RTT, MTU, iperf3),
2. shows what each endpoint advertises,
3. transfers Cape Town -> DTN and DTN -> Cape Town, sampling the DTN's sockets while it runs, and reports which local address carried the data.

## 1. What the weave built

In [ ]:
import json, os, re, time
from fabric_env import get_fablib, load_legs

CAPE_COLLECTION = "35df6f87-df63-11f1-977e-02ffe3faed53"   # SANReN GCP - Cape Town SC26 (cryoem-without-borders)
CAPE_SRC_PATH   = "/"              # what to pull from Cape Town (file or directory in the guest collection)
CAPE_DST_DIR    = "/from-fabric/"  # where pushes land on the Cape Town collection (needs R/W there)
PUSH_SIZE_GB    = 10               # size of the generated file for the DTN -> Cape Town run

built = load_legs()
legs = {s: l for s, l in built["legs"].items() if "error" not in l}
for s, l in built["legs"].items():
    if "error" in l:
        print(f"{s}: FAILED in the weave - {l['error'].splitlines()[0]}")
    else:
        print(f"{s}: data {l['data_addr']} on {l.get('l2_fp')} VLAN {l.get('l2_vlan')} (MTU {l['mtu']}) -> {l['peer_ip']}; public {l.get('public_ip', '-')}; "
              f"Globus {l.get('globus')}; collection {l.get('collection_id', '-')}")

fablib = get_fablib()
def node(site):
    return fablib.get_slice(name=legs[site]["slice"]).get_node(name="dtn")

## 2. The data path, from each DTN

`ip route get` shows which interface the DTN would use toward Cape Town. It must be the data NIC, not the
management NIC (`eth0`) or the public uplink. Cape Town answers only once the operators have mapped the FABRIC
VLAN to the Cape Town VLAN on the switch.

In [ ]:
for site, l in legs.items():
    n = node(site)
    out, _ = n.execute(
        f"ip route get {l['peer_ip']} | head -1; "
        f"ping -c 5 -q {l['peer_ip']} | tail -2; "
        f"ping -c 2 -W 2 -M do -s {l['mtu'] - 28} {l['peer_ip']} >/dev/null && echo 'MTU {l['mtu']} passes' "
        f"|| echo 'MTU {l['mtu']} does NOT pass'", quiet=True)
    print(f"--- {site} ({l['data_dev']} {l['data_addr']})\n{out}")

### iperf3 baseline

Johann has opened TCP/UDP 5201 on the Cape Town DTN. If no iperf3 server is running there, this says so. Ask them to
run `iperf3 -s` and try again.

In [ ]:
for site, l in legs.items():
    out, _ = node(site).execute(
        f"timeout 40 iperf3 -c {l['peer_ip']} -B {l['data_ip']} -P 8 -t 20 -f g 2>&1 | tail -4", quiet=True)
    print(f"--- {site} -> Cape Town\n{out}")

## 3. What each endpoint advertises

`node list` shows the addresses Globus registered for the node and the **data interface** it hands to the other
side. Control goes to the public IP; data goes to the data interface.

In [ ]:
for site, l in legs.items():
    if l.get("globus") != "ready":
        print(f"{site}: Globus not registered ({l.get('globus')})"); continue
    out, _ = node(site).execute(
        "sudo bash -c '. /etc/globus-dtn/creds.env && export GCS_CLI_CLIENT_ID GCS_CLI_CLIENT_SECRET "
        "GCS_CLI_ENDPOINT_ID=$(cat /etc/globus-dtn/endpoint-id.txt) && globus-connect-server node list'",
        quiet=True)
    print(f"--- {site}: public {l['public_ip']}, data {l['data_ip']}\n{out}")

## 4. Sign in to Globus

The demo needs a Globus identity that is **both** in the weave's `GLOBUS_USERS` (the DTN collections) **and**
granted on the Cape Town guest collection by SANReN. Open the link, sign in, and paste the code back.
`GLOBUS_NATIVE_CLIENT_ID` defaults to the Globus SDK tutorial client. Register your own native app at
app.globus.org > Settings > Developers for anything long-lived.

In [ ]:
import globus_sdk

CLIENT_ID = os.environ.get("GLOBUS_NATIVE_CLIENT_ID", "61338d24-54d5-408f-a10d-66c06b59f6d2")
mapped = [l["collection_id"] for l in legs.values() if l.get("collection_id")]
# GCS 5 mapped collections need a data_access consent each; the Cape Town GCP collection does not.
scope = "urn:globus:auth:scope:transfer.api.globus.org:all"
if mapped:
    scope += "[" + " ".join(f"*https://auth.globus.org/scopes/{c}/data_access" for c in mapped) + "]"

auth = globus_sdk.NativeAppAuthClient(CLIENT_ID)
auth.oauth2_start_flow(requested_scopes=scope)
print("Sign in here:\n", auth.oauth2_get_authorize_url())
code = input("Authorization code: ").strip()
tok = auth.oauth2_exchange_code_for_tokens(code).by_resource_server["transfer.api.globus.org"]
tc = globus_sdk.TransferClient(authorizer=globus_sdk.AccessTokenAuthorizer(tok["access_token"]))
print("signed in")

In [ ]:
print(f"Cape Town collection {CAPE_COLLECTION}, {CAPE_SRC_PATH}:")
for e in tc.operation_ls(CAPE_COLLECTION, path=CAPE_SRC_PATH):
    print(f"  {e['type']:4s} {e['size']:>15,d}  {e['name']}")

## 5. Transfer and watch the sockets

`transfer_and_watch` submits a transfer, then samples the DTN's TCP connections on the GridFTP data ports every few
seconds while it runs. Each data connection is listed with its local and remote address and the bytes it carried.
The verdict compares the local address with the DTN's data IP and public IP. Use files large enough to run for at
least 15-20 s. Small files can finish between two samples.

In [ ]:
SS = ("ss -Htni state established "
      "'( sport >= :50000 and sport <= :51000 ) or ( dport >= :50000 and dport <= :51000 )'")

def _ip(ep):
    return ep.rsplit(":", 1)[0].strip("[]").replace("::ffff:", "")

def data_conns(n):
    conns, cur = {}, None
    for line in n.execute(SS, quiet=True)[0].splitlines():
        if not line.strip():
            continue
        if not line[0].isspace():
            f = line.split()
            cur = (f[2], f[3]) if len(f) >= 4 else None
            if cur: conns[cur] = 0
        elif cur:
            got = [int(x) for x in re.findall(r"bytes_(?:received|acked):(\d+)", line)]
            conns[cur] = max(got or [0])
    return conns

def transfer_and_watch(site, src, src_path, dst, dst_path, label, recursive=False, every=3):
    l, n = legs[site], node(site)
    td = globus_sdk.TransferData(source_endpoint=src, destination_endpoint=dst, label=label,
                                 verify_checksum=True)
    td.add_item(src_path, dst_path, recursive=recursive)
    task = tc.submit_transfer(td)["task_id"]
    print(f"task {task}: {src_path} -> {dst_path}")
    seen = {}
    while True:
        for k, b in data_conns(n).items():
            seen[k] = max(seen.get(k, 0), b)
        t = tc.get_task(task)
        print(f"  {t['status']:9s} {t.get('nice_status') or '':12s} {t['bytes_transferred']:>15,d} B  "
              f"{len(seen)} data conn(s) seen", end="\r")
        if t["status"] in ("SUCCEEDED", "FAILED") or (t["status"] == "ACTIVE" and t.get("nice_status") in
                                                        ("PERMISSION_DENIED", "ENDPOINT_ERROR")):
            break
        time.sleep(every)
    print()
    by_pair = {}
    for (loc, rem), b in seen.items():
        k = (_ip(loc), _ip(rem)); by_pair[k] = by_pair.get(k, 0) + b
    for (loc, rem), b in sorted(by_pair.items()):
        tag = ("<- data NIC (Cape Town path)" if loc == l["data_ip"] else
               "<- PUBLIC uplink" if loc == l.get("public_ip") else "<- other")
        print(f"  {loc:>15s} <-> {rem:15s} {b:>15,d} B  {tag}")
    if not by_pair:
        print("  no data connection was caught - use a larger file (or a smaller `every`).")
    elif all(loc == l["data_ip"] for loc, _ in by_pair):
        print(f"  VERDICT: all data went over {l['data_ip']} - the Cape Town path.")
    else:
        print("  VERDICT: some data did NOT use the data NIC - check `node list` (section 3).")
    rate = t.get("effective_bytes_per_second") or 0
    print(f"  {t['status']}: {t['bytes_transferred']:,d} B, {rate * 8 / 1e9:.2f} Gb/s effective")
    return task

### 5a. Cape Town -> DTN

In [ ]:
for site, l in legs.items():
    if l.get("globus") != "ready":
        continue
    acct = next(iter(l["users"].values()))           # first mapped account; its private directory
    is_dir = CAPE_SRC_PATH.endswith("/")
    transfer_and_watch(site, CAPE_COLLECTION, CAPE_SRC_PATH, l["collection_id"],
                       f"/{acct}/from-cape/" + ("" if is_dir else os.path.basename(CAPE_SRC_PATH)),
                       f"Cape Town to {site} (SC26)", recursive=is_dir)

### 5b. DTN -> Cape Town

This needs write access on the Cape Town collection. It generates a `PUSH_SIZE_GB` file in the mapped account's
directory on the DTN and pushes it to `CAPE_DST_DIR`.

In [ ]:
try:
    tc.operation_mkdir(CAPE_COLLECTION, CAPE_DST_DIR)
except globus_sdk.TransferAPIError as e:
    if "Exists" not in str(e.code):
        raise
for site, l in legs.items():
    if l.get("globus") != "ready":
        continue
    acct = next(iter(l["users"].values()))
    fname = f"sc26-{site.lower()}-{PUSH_SIZE_GB}G.bin"
    node(site).execute(f"sudo -u {acct} bash -c 'f={l['data_base']}/{acct}/{fname}; "
                       f"[ -s $f ] || head -c {PUSH_SIZE_GB}G /dev/urandom > $f'", quiet=True)
    transfer_and_watch(site, l["collection_id"], f"/{acct}/{fname}", CAPE_COLLECTION,
                       CAPE_DST_DIR + fname, f"{site} to Cape Town (SC26)")

## 6. The same thing from the Globus CLI

These commands take exactly the same path. Submitting from the CLI, the web app, or this notebook makes no
difference, because the route is set by the endpoints' advertised data interfaces.

In [ ]:
for site, l in legs.items():
    if l.get("collection_id"):
        acct = next(iter(l["users"].values()))
        print(f"# {site}")
        print(f"globus transfer {CAPE_COLLECTION}:{CAPE_SRC_PATH} {l['collection_id']}:/{acct}/from-cape/ "
              f"--recursive --verify-checksum --label 'Cape Town to {site}'")
        print(f"# web: https://app.globus.org/file-manager?origin_id={CAPE_COLLECTION}"
              f"&destination_id={l['collection_id']}\n")